# SC203 DoSSR — LoRA 2 vs. LoRA 3 Controlled Colab Pilot & Bicubic Floor (`03_lora_pilot_colab.ipynb`)

**Team:** LEMMEPASS (Team 1, SC203 Scientific Methods — Class 25A02)  
**Target Branch:** `feat/baseline-runner`  
**Purpose:** End-to-end Google Colab L4 GPU notebook for:
1. Environment / GPU / Drive verification + **Bicubic $\times 4$ whole-image floor** (`RealSRVal_crop128`) for Assignment 5 Table 1.
2. **Scene-level Real-CE $\times 4$ `train` split & patch extraction** (`scripts/split_realce_by_scene.py`, zero scene leakage, `test` held out untouched).
3. **LoRA 2 pilot training** (text-weighted reconstruction loss, $\lambda_{\text{cons}} = 0$).
4. **LoRA 3 pilot training** (same init, data, empty prompt `""`, $K=2$ noise draws, `seed=231`, 400 steps + cross-noise consistency).
5. **Fixed-validation 10-seed (`42..51`), 5-step inference** (`per_image_seed: true`) + full-reference metrics + RapidOCR evaluation.
6. **Decision gate & mentor-update table** (delta Error Rate, CER, Seed Disagreement, LPIPS, Peak VRAM GiB, and mean/p95 `s/step`).

## CELL 01 — Mount Google Drive, Pull Branch & Check L4 GPU / Dependencies

In [ ]:
# === CELL 01: Mount Drive, clone/pull SC203, verify L4 GPU & dependencies ===
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = Path('/content/SC203')
DRIVE_BASE = Path('/content/drive/MyDrive/lemmepass')
DATA_DIR = DRIVE_BASE / 'data'
OUT_BASE = DRIVE_BASE / 'outputs'
CKPT_PATH = DRIVE_BASE / 'weights' / 'dossr_default.ckpt'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', 'feat/baseline-runner', 'https://github.com/lemon4life/SC203.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', 'feat/baseline-runner'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/feat/baseline-runner'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)

dossr_cfg = REPO_DIR / 'external' / 'DOSSR' / 'configs' / 'model' / 'cldm_v21.yaml'
REQUIRED_DOSSR_SPECS = ('pytorch_lightning', 'open_clip', 'omegaconf', 'einops', 'transformers', 'timm')
missing_dossr = [pkg for pkg in REQUIRED_DOSSR_SPECS if importlib.util.find_spec(pkg) is None]
if not dossr_cfg.is_file() or missing_dossr:
    print(f'Bootstrapping external/DOSSR (cfg={dossr_cfg.is_file()}, missing={missing_dossr}) via scripts/colab_setup.py...')
    if importlib.util.find_spec('jedi') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jedi==0.20.1'], check=False)
    setup_proc = subprocess.Popen(
        [sys.executable, '-u', 'scripts/colab_setup.py', '--allow-preexisting-conflicts'],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert setup_proc.stdout is not None
    for _line in setup_proc.stdout:
        print(_line, end='', flush=True)
    _rc = setup_proc.wait()
    if _rc != 0:
        raise RuntimeError(f'scripts/colab_setup.py failed with exit code {_rc}')

still_missing = [pkg for pkg in REQUIRED_DOSSR_SPECS if importlib.util.find_spec(pkg) is None]
if still_missing or not dossr_cfg.is_file():
    raise RuntimeError(f'DOSSR setup incomplete (cfg={dossr_cfg.is_file()}, missing={still_missing})')

missing_pkgs = [pkg for pkg in ('rapidocr_onnxruntime', 'lpips') if importlib.util.find_spec(pkg) is None]
if missing_pkgs:
    print('Installing missing packages:', missing_pkgs)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_pkgs], check=True)

import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.2f} GiB')
print('DOSSR config exists:', dossr_cfg.is_file(), '| Checkpoint exists:', CKPT_PATH.is_file(), '| path:', CKPT_PATH)

## CELL 02 — CPU Sanity Verification (`src/lora_text_loss.py` & `scripts/run_lora_trial.py`)

In [ ]:
# === CELL 02: Run offline CPU unit tests & training pipeline dry run ===
import subprocess
import sys

def run_streamed(cmd):
    print('RUN:', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Command failed with exit code {rc}')

run_streamed([sys.executable, '-u', 'src/lora_text_loss.py', '--run-cpu-tests'])
run_streamed([sys.executable, '-u', 'scripts/run_lora_trial.py', '--dry-run-cpu'])

## CELL 03 — Fill Assignment 5 Table 1: Bicubic $\times 4$ Whole-Image Floor (`RealSRVal_crop128`)

In [ ]:
# === CELL 03: Compute Bicubic x4 whole-image PSNR_RGB, SSIM_RGB, PSNR_Y, SSIM_Y, LPIPS on RealSRVal_crop128 ===
_realsr_candidates = [
    DATA_DIR / 'StableSR_testsets' / 'StableSR_testsets' / 'RealSRVal_crop128',
    DATA_DIR / 'RealSRVal_crop128',
    Path('/content/data_local/StableSR_testsets/RealSRVal_crop128'),
    Path('/content/data_local/RealSRVal_crop128'),
]
REALSR_ROOT = next(
    (p for p in _realsr_candidates if (p / 'test_LR').is_dir() and (p / 'test_HR').is_dir()),
    None,
)
BICUBIC_SUMMARY_CSV = REPO_DIR / 'results' / 'metrics_realsr_bicubic_x4_summary.csv'

if REALSR_ROOT is None:
    print('[SKIP] RealSRVal_crop128 (test_LR / test_HR) not found under DATA_DIR or /content/data_local; skipping optional Table 1 Bicubic floor.')
else:
    print(f'Found RealSRVal_crop128 at: {REALSR_ROOT}')
    run_streamed([
        sys.executable, '-u', 'scripts/compute_metrics.py',
        '--bicubic-lr-dir', str(REALSR_ROOT / 'test_LR'),
        '--gt-dir', str(REALSR_ROOT / 'test_HR'),
        '--expected-size', '512',
        '--device', 'cuda',
        '--summary-csv', str(BICUBIC_SUMMARY_CSV),
    ])

## CELL 04 — Scene-Level Real-CE $\times 4$ `train` Split & Aligned Patch Extraction (Zero Leakage)

In [ ]:
# === CELL 04: Stage Real-CE on fast local SSD if zipped, split train by scene (val held out), & extract aligned 128->512 patches ===
import subprocess
import zipfile

sys.path.insert(0, str(REPO_DIR))
from scripts.check_realce_feasibility import resolve_realce_root

LOCAL_REALCE = Path('/content/data_local/Real-CE')
DRIVE_REALCE = DATA_DIR / 'Real-CE'
ZIP_CANDIDATES = [Path('/content/Real-CE.zip'), DATA_DIR / 'Real-CE.zip']

def _has_realce(path: Path) -> bool:
    r = resolve_realce_root(path)
    return (r / 'train' / '13mm').is_dir() and (r / 'train' / '52mm').is_dir()

if _has_realce(LOCAL_REALCE):
    REALCE_DIR = resolve_realce_root(LOCAL_REALCE)
elif any(z.is_file() for z in ZIP_CANDIDATES):
    src_zip = next(z for z in ZIP_CANDIDATES if z.is_file())
    LOCAL_REALCE.mkdir(parents=True, exist_ok=True)
    print(f'Unpacking {src_zip} to fast local SSD ({LOCAL_REALCE})...')
    rc = subprocess.run(['unzip', '-q', '-o', str(src_zip), '-d', str(LOCAL_REALCE)], check=False).returncode
    if rc != 0:
        with zipfile.ZipFile(src_zip, 'r') as zf:
            zf.extractall(LOCAL_REALCE)
    REALCE_DIR = resolve_realce_root(LOCAL_REALCE)
else:
    REALCE_DIR = resolve_realce_root(DRIVE_REALCE)

REALCE_PATCH_DIR = Path('/content/data_local/RealCE_patches')
print(f'Using REALCE_DIR       : {REALCE_DIR}')
print(f'Writing REALCE_PATCH_DIR: {REALCE_PATCH_DIR}')

run_streamed([
    sys.executable, '-u', 'scripts/split_realce_by_scene.py',
    '--data-dir', str(REALCE_DIR),
    '--split', 'train',
    '--scale', 'x4',
    '--val-ratio', '0.20',
    '--seed', '42',
    '--output-manifest', 'data/realce_splits_manifest.json',
    '--output-train-csv', 'data/realce_train_manifest.csv',
    '--output-train-boxes', 'data/realce_train_boxes.csv',
    '--output-val-boxes', 'data/realce_val_boxes.csv',
    '--extract-patches',
    '--patch-dir', str(REALCE_PATCH_DIR),
    '--hr-patch-size', '512',
    '--lr-patch-size', '128',
    '--stride', '256',
    '--max-patches-per-scene', '2',
    '--max-val-patches', '30',
])

import csv as _csv
with open('data/realce_train_manifest.csv', encoding='utf-8') as _f:
    _mrows = list(_csv.DictReader(_f))
with open('data/realce_train_boxes.csv', encoding='utf-8') as _f:
    _tboxes = sum(1 for _ in _csv.DictReader(_f))
with open('data/realce_val_boxes.csv', encoding='utf-8') as _f:
    _vboxes = sum(1 for _ in _csv.DictReader(_f))
_tr_p = sum(1 for r in _mrows if r.get('split') == 'train')
_va_p = sum(1 for r in _mrows if r.get('split') == 'val')
print(f'Real-CE patch summary -> train patches: {_tr_p} (boxes: {_tboxes}) | val patches: {_va_p} (boxes: {_vboxes})')

## CELL 05 — Train Controlled Pilot: LoRA 2 (Text-Weighted L1) vs. LoRA 3 (+ Cross-Noise Consistency)

In [ ]:
# === CELL 05: Train LoRA 2 and LoRA 3 under identical init, data, empty prompt "", K=2 noise draws, seed=231, 400 steps ===
LORA2_OUT = OUT_BASE / 'lora2-pilot'
LORA3_OUT = OUT_BASE / 'lora3-pilot'

run_streamed([
    sys.executable, '-u', 'scripts/run_lora_trial.py',
    '--config', 'configs/lora2_pilot.yaml',
    '--train-manifest', 'data/realce_train_manifest.csv',
    '--output-dir', str(LORA2_OUT),
])

run_streamed([
    sys.executable, '-u', 'scripts/run_lora_trial.py',
    '--config', 'configs/lora3_pilot.yaml',
    '--train-manifest', 'data/realce_train_manifest.csv',
    '--output-dir', str(LORA3_OUT),
])

## CELL 06 — Fixed-Validation 10-Seed (`42..51`), 5-Step Inference & Quantitative Evaluation

In [ ]:
# === CELL 06: 10-seed 5-step validation inference + LPIPS/PSNR/SSIM + OCR comparison ===
import yaml

VAL_LORA2_OUT = OUT_BASE / 'pilot-lora2-realce-val-10seed'
VAL_LORA3_OUT = OUT_BASE / 'pilot-lora3-realce-val-10seed'
LORA2_WEIGHTS = (LORA2_OUT if 'LORA2_OUT' in globals() else OUT_BASE / 'lora2-pilot') / 'lora_weights_final.pt'
LORA3_WEIGHTS = (LORA3_OUT if 'LORA3_OUT' in globals() else OUT_BASE / 'lora3-pilot') / 'lora_weights_final.pt'

def _write_val_runtime_cfg(src_yaml: Path, dst_yaml: Path, *, out_root: Path, lora_weights: Path) -> Path:
    cfg = yaml.safe_load(src_yaml.read_text(encoding='utf-8'))
    cfg['input_dir'] = str(REALCE_PATCH_DIR / 'val' / 'LR')
    cfg['gt_dir'] = str(REALCE_PATCH_DIR / 'val' / 'HR')
    cfg['output_root'] = str(out_root)
    cfg['lora_weights'] = str(lora_weights)
    cfg['baseline_config'] = str(dst_yaml)
    dst_yaml.write_text(yaml.safe_dump(cfg), encoding='utf-8')
    return dst_yaml

lora2_val_cfg = _write_val_runtime_cfg(
    REPO_DIR / 'configs' / 'lora2_pilot.yaml',
    Path('/content/lora2_val_runtime.yaml'),
    out_root=VAL_LORA2_OUT,
    lora_weights=LORA2_WEIGHTS,
)
lora3_val_cfg = _write_val_runtime_cfg(
    REPO_DIR / 'configs' / 'lora3_pilot.yaml',
    Path('/content/lora3_val_runtime.yaml'),
    out_root=VAL_LORA3_OUT,
    lora_weights=LORA3_WEIGHTS,
)

run_streamed([sys.executable, '-u', 'scripts/run_seed_study.py', '--config', str(lora2_val_cfg)])
run_streamed([sys.executable, '-u', 'scripts/run_seed_study.py', '--config', str(lora3_val_cfg)])

for label, study_out in [('lora2', VAL_LORA2_OUT), ('lora3', VAL_LORA3_OUT)]:
    run_streamed([
        sys.executable, '-u', 'scripts/compute_metrics.py',
        '--output-root', str(study_out),
        '--gt-dir', str(REALCE_PATCH_DIR / 'val' / 'HR'),
        '--expected-size', '512',
        '--test-y-channel',
        '--summary-csv', str(REPO_DIR / 'results' / f'metrics_pilot_{label}_val_summary.csv'),
        '--per-image-csv', str(REPO_DIR / 'results' / f'metrics_pilot_{label}_val_per_seed.csv'),
    ])

run_streamed([
    sys.executable, '-u', 'scripts/text_eval.py',
    '--study-root', str(VAL_LORA2_OUT),
    '--compare-study-root', str(VAL_LORA3_OUT),
    '--hr-dir', str(REALCE_PATCH_DIR / 'val' / 'HR'),
    '--lr-dir', str(REALCE_PATCH_DIR / 'val' / 'LR'),
    '--boxes-csv', 'data/realce_val_boxes.csv',
    '--ocr-backend', 'rapidocr',
    '--rec-only',
    '--rotate-vertical',
    '--upscale', '2',
    '--output-dir', 'results/pilot_lora2_vs_lora3_reconly',
])

run_streamed([
    sys.executable, '-u', 'scripts/text_eval.py',
    '--study-root', str(VAL_LORA3_OUT),
    '--boxes-csv', 'data/realce_val_boxes.csv',
    '--ocr-backend', 'rapidocr',
    '--rec-only',
    '--rotate-vertical',
    '--upscale', '2',
    '--output-dir', 'results/pilot_lora3_reconly',
])

## CELL 07 — Pilot Decision Gate & Mentor Update Summary (VRAM & `s/step`)

In [ ]:
# === CELL 07: Print Peak VRAM, step time, LPIPS/seed stability, and OCR comparison table for Dr. Thuan update ===
import csv
import json

def _mean_col(csv_path: Path, col: str) -> tuple[float, int] | None:
    if not csv_path.is_file():
        return None
    rows = list(csv.DictReader(csv_path.open(encoding='utf-8')))
    if not rows or col not in rows[0]:
        return None
    vals = [float(r[col]) for r in rows]
    return sum(vals) / len(vals), len(vals)

print('--- Training Cost & Peak VRAM ---')
for name, out_dir in [('LoRA 2', LORA2_OUT), ('LoRA 3', LORA3_OUT)]:
    m_path = out_dir / 'lora_run_manifest.json'
    if m_path.is_file():
        m = json.loads(m_path.read_text(encoding='utf-8'))
        print(
            f"{name:8s} | params={m['trainable_lora_params']:,} | "
            f"mean={m['mean_step_time_sec']:.2f}s/step (p95={m['p95_step_time_sec']:.2f}s) | "
            f"peak_alloc={m['peak_vram_allocated_gib']:.2f} GiB | "
            f"peak_res={m['peak_vram_reserved_gib']:.2f} GiB"
        )

print('\n--- Whole-Patch Reconstruction & 10-Seed Stability (Real-CE Val) ---')
for label in ('lora2', 'lora3'):
    summary_csv = REPO_DIR / 'results' / f'metrics_pilot_{label}_val_summary.csv'
    if summary_csv.is_file():
        rows = list(csv.DictReader(summary_csv.open(encoding='utf-8')))
        overall = next((r for r in rows if r['image_stem'] == 'ALL_IMAGES_MEAN'), None)
        if overall:
            print(
                f"{label.upper():8s} VAL | PSNR_Y={float(overall['psnr_y_mean']):.4f} | "
                f"SSIM_Y={float(overall['ssim_y_mean']):.4f} | LPIPS={float(overall['lpips_mean']):.4f} | "
                f"pair_px={float(overall['pairwise_seed_px_mean_abs_diff']):.4f} | "
                f"pair_lpips={float(overall['pairwise_seed_lpips_mean']):.4f}"
            )

print('\n--- Text-Box OCR Comparison (RapidOCR, Real-CE Val) ---')
ocr_dir_pair = REPO_DIR / 'results' / 'pilot_lora2_vs_lora3_reconly'
ocr_dir_l3 = REPO_DIR / 'results' / 'pilot_lora3_reconly'
if not ocr_dir_pair.is_dir():
    ocr_dir_pair = REPO_DIR / 'results' / 'pilot_lora2_vs_lora3'
if not ocr_dir_l3.is_dir():
    ocr_dir_l3 = REPO_DIR / 'results' / 'pilot_lora3_val_text_eval'
for tag, p in [
    ('HR Ceiling', ocr_dir_pair / 'text_eval_hr_ceiling_by_image.csv'),
    ('Bicubic x4', ocr_dir_pair / 'text_eval_bicubic_floor_by_image.csv'),
    ('LoRA 2 VAL', ocr_dir_pair / 'text_eval_summary_by_image.csv'),
    ('LoRA 3 VAL', ocr_dir_l3 / 'text_eval_summary_by_image.csv'),
]:
    wc = _mean_col(p, 'wrong_char_rate')
    cer = _mean_col(p, 'cer_mean')
    dis = _mean_col(p, 'ocr_seed_disagreement_rate')
    if wc and cer:
        extra = f' | seed_disagree={dis[0]:.4f}' if (dis and 'LoRA' in tag) else ''
        print(f"{tag:10s} (n={wc[1]:2d}) | wrong_char={wc[0]:.4f} | CER={cer[0]:.4f}{extra}")

delta_csv = ocr_dir_pair / 'text_eval_paired_comparison_by_image.csv'
dw = _mean_col(delta_csv, 'delta_wrong_char_rate_compare_minus_base')
dc = _mean_col(delta_csv, 'delta_cer_compare_minus_base')
if dw and dc:
    print(f'Paired (LoRA 3 - LoRA 2, n={dw[1]}): delta_wrong_char={dw[0]:+.4f} | delta_CER={dc[0]:+.4f}')